# 04 — Tokens, Context Windows, Logits, and Decoding

**Network LLM Engineering — Part I — Foundations**

### Learning goals
- Inspect real model tokenization
- Understand next-token probabilities
- Compare deterministic and stochastic decoding

In [ ]:
%pip install -q transformers==5.14.1 datasets==5.0.1 accelerate==1.14.0 peft==0.20.0 trl==1.10.0 sentence-transformers==5.7.0 pandas matplotlib scikit-learn requests jsonschema

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
samples = [
    "OSPF neighbor stuck in EXSTART",
    "show bgp l2vpn evpn route-type 2",
    "Ethernet1/49 CRC errors increasing",
]
for s in samples:
    ids = tok(s)["input_ids"]
    print("\nTEXT:", s)
    print("TOKENS:", len(ids), ids)
    print([tok.decode([i]) for i in ids])

## Context window is not memory

The **context window** is the token sequence available to the model for the current inference.
It is not the same as long-term memory, a database, or learned weights.

Putting 100,000 tokens into context may be technically possible for some models, but it is not automatically a good retrieval strategy.
Long contexts cost compute, can dilute relevance, and still require evaluation.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
    torch.float16 if torch.cuda.is_available() else torch.float32
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
).eval()

def render_chat(messages, generation=True):
    kw = dict(tokenize=False, add_generation_prompt=generation)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)

@torch.inference_mode()
def generate(messages, max_new_tokens=160, temperature=0.0):
    text = render_chat(messages, True)
    toks = tokenizer(text, return_tensors="pt")
    dev = next(model.parameters()).device
    toks = {k:v.to(dev) for k,v in toks.items()}
    sample = temperature > 0
    out = model.generate(
        **toks, max_new_tokens=max_new_tokens,
        do_sample=sample,
        temperature=temperature if sample else None,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0, toks["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
prompt = [{"role":"user","content":"Give one likely cause of OSPF EXSTART and one verification step."}]
for temp in [0.0, 0.7, 1.1]:
    print("\nTemperature", temp)
    print(generate(prompt, max_new_tokens=80, temperature=temp))

## Decoding terminology

- **Greedy:** choose the highest-probability token each step.
- **Temperature:** rescales logits; higher usually increases randomness.
- **Top-k:** sample from the k most probable tokens.
- **Top-p:** sample from a probability mass cutoff.
- **Reasoning/test-time compute:** can involve more generated/search computation, but is separate from decoding temperature.

### Exercise

Run the same troubleshooting prompt five times at temperature `1.0`.
Would you want this amount of variation in an automated NOC decision path?